In [ ]:
from circuitcost.data import DATASET_SUBDIR, find_project_root, load_designs

project_root = find_project_root()

Install dependencies before opening the notebook: run `make setup` and `make notebooks` from the project root. Select that environment as the kernel.

#Loading the data and checking the datatypes

In [ ]:
import pandas as pd

In [ ]:
data_dir = project_root / DATASET_SUBDIR

hls = pd.read_csv(data_dir / "post_hls_info.csv")
impl = pd.read_csv(data_dir / "post_implementation_info.csv")

# Explicit one-to-one prj join, with checks for unmatched and duplicate IDs.
# The loader preserves HLS file order so the recorded random split is reproducible.
designs = load_designs(data_dir)
display(designs.head())

In [ ]:
hls.head()

In [ ]:
impl.head()

In [ ]:
print("HLS rows and columns:", hls.shape)

In [ ]:
print("Implementation rows and columns", impl.shape)

In [ ]:
print("HLS columns:")
#print(*hls.columns, sep="\n")
print(hls.dtypes)

In [ ]:
print("Implementation columns:")
#print(*impl.columns, sep="\n")
print(impl.dtypes)

#Checking wether data is complete or not

In [ ]:
#Checking missing values
print(hls.isna().sum())
#No missing values found

In [ ]:
print(impl.isna().sum())
#No missing values found

In [ ]:
display(hls.head(10))

In [ ]:
display(impl.head(10))

In [ ]:
#Finding columns shared by both tables
common_columns = hls.columns.intersection(impl.columns)
display(common_columns)

In [ ]:
#Now Understand what the candidate represents
print(hls["prj"].head())
print(impl["prj"].head())

In [ ]:
#Check completeness and Uniqueness
for name,table in [("HLS",hls),("Implementation",impl)]:
    print(name)
    print("Missing IDs:", table["prj"].isna().sum())
    print("Duplicate IDs:", table["prj"].duplicated().sum())

In [ ]:
#Check that Identifiers match across tables
hls_ids = set(hls["prj"])
impl_ids = set(impl["prj"])

print("Only in HLS", len(hls_ids - impl_ids))
print("Only in IMPL", len(impl_ids - hls_ids))

In [ ]:
#Join with checks enabled
combined = hls.merge(
    impl,
    on="prj",
    how="outer",
    validate="one_to_one",
    indicator=True,
)

print(combined["_merge"].value_counts())

In [ ]:
#Dropping merge as it is not required anymore
combined = combined.drop(columns=["_merge"])

In [ ]:
#Calculating min, max and mean of lut and Total LUTs column from hls and impl table respectively
min_hls = min(hls["lut"])
print("Minimum lut count in HLS",min_hls)
max_hls = max(hls["lut"])
print("Maximum lut count in HLS",max_hls)
min_impl = min(impl["Total LUTs"])
print("Minimum lut count in impl",min_impl)
max_impl = max(impl["Total LUTs"])
print("Maximum lut count in impl",max_impl)
mean_hls_lut = (sum(hls["lut"])/len(hls["lut"]))
print("Mean of HLS estimated LUT", mean_hls_lut)
mean_impl_total_luts = (sum(impl["Total LUTs"])/len(impl["Total LUTs"]))
print("Mean of IMPL Total LUTs count",mean_impl_total_luts)

In [ ]:
#Calculating median and standard deviation
median_hls = hls["lut"].median()
median_impl = impl['Total LUTs'].median()

print("Median HLS LUTs",median_hls)
print("Median Impl LUTs", median_impl)

std_hls = hls["lut"].std()
std_impl = impl["Total LUTs"].std()

print("Standard Deviation HLS luts", std_hls)
print("Standard Deviation Impl luts", std_impl)

Install dependencies before opening the notebook: run `make setup` and `make notebooks` from the project root. Select that environment as the kernel.

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.figure(figsize=(8,4))

plt.hist(
    hls["lut"].dropna(),
    bins=50,
    edgecolor="black"
)

plt.xlabel("Estimated LUT count")
plt.ylabel("Number of designs")
plt.title("Distribution of HLS LUT usage")

#Marking mean and median
plt.axvline(
    hls["lut"].mean(),
    color="red",
    linestyle="--",
    label = "Mean"
)

plt.axvline(
    hls["lut"].median(),
    color="green",
    linestyle = "--",
    label = "Median"
)
#adding minimum and maximum
plt.axvline(
    hls["lut"].min(),
    color="blue",
    linestyle=":",
    label = f'Min: {hls["lut"].min():,.0f}'
)
plt.axvline(
    hls["lut"].max(),
    color="purple",
    linestyle=":",
    label = f'Max: {hls["lut"].max():,.0f}'
)

plt.axvline(
    hls["lut"].std(),
    color="pink",
    linestyle=":",
    label = f'Std: {hls["lut"].std():,.0f}'
)
plt.legend()
plt.show()

In [ ]:
#plot every numerical column
hls.select_dtypes(include="number").hist(
    bins=30,
    figsize=(16, 12),
    edgecolor="black"
)

plt.tight_layout()
plt.show()

In [ ]:
#Repeating smae for impl dataset

plt.figure(figsize=(8,4))

plt.hist(
    impl["Total LUTs"].dropna(),
    bins=50,
    edgecolor = "black"
)

plt.xlabel("Total LUTs used")
plt.ylabel("Number of designs")
plt.title("Distribution of impl LUT count")

plt.axvline(
    impl["Total LUTs"].mean(),
    color="red",
    linestyle="--",
    label="mean"
)

plt.axvline(
    impl["Total LUTs"].median(),
    color="green",
    linestyle = "--",
    label="median"
)

#Adding Minimum and Maximum

plt.axvline(
    impl["Total LUTs"].min(),
    color="blue",
    linestyle = ":",
    label=f'Min: {impl["Total LUTs"].min():,.0f}'
)

plt.axvline(
    impl["Total LUTs"].max(),
    color="blue",
    linestyle = ":",
    label=f'Max: {impl["Total LUTs"].max():,.0f}'
)

plt.axvline(
    impl["Total LUTs"].std(),
    color="pink",
    linestyle = ":",
    label=f'Std: {impl["Total LUTs"].std():,.0f}'
)

plt.legend()
plt.show()

In [ ]:
#Checking all other columns of impl dataset

impl.select_dtypes(include="number").hist(
    bins=50,
    figsize=(16,12),
    edgecolor="black"
)
plt.tight_layout()
plt.show()

In [ ]:
paired = hls[["prj","lut"]].merge(
    impl[["prj","Total LUTs"]],
    on="prj",
    how="inner",
    validate="one_to_one"
)

plt.figure(figsize=(8,6))

plt.scatter(
    paired["lut"],
    paired["Total LUTs"],
    alpha = 0.4
)

plt.xlabel("HLS estimated LUT count")
plt.ylabel("Implementation Total LUTs")
plt.title("HLS estimates vs implementation LUT usage")
plt.grid(alpha=0.9)

lower = min(paired["lut"].min(), paired["Total LUTs"].min())
upper = max(paired["lut"].max(), paired["Total LUTs"].max())

plt.plot(
    [lower,upper],
    [lower, upper],
    color="red",
    linestyle="--",
    label="y=x"
)
plt.legend()
plt.show()